<a href="https://colab.research.google.com/github/tarunmikkilineni/dataviz-exercises-TarunMikkilineni/blob/main/week07.lecture07_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:

# Task 1 — Filter fossil fuel data

df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Display the first 10 rows
df_fossil.head(10)


,Country,Region,Source,Share_pct,TWh,Source_Type
0,United States,North America,Coal,10,1015.0,Fossil
1,United States,North America,Oil,35,3220.0,Fossil
2,United States,North America,Natural Gas,34,3083.0,Fossil
8,China,Asia,Coal,58,6929.1,Fossil
9,China,Asia,Oil,18,1620.0,Fossil
10,China,Asia,Natural Gas,10,872.0,Fossil
15,Germany,Europe,Coal,5,537.0,Fossil
16,Germany,Europe,Oil,15,1361.0,Fossil
17,Germany,Europe,Natural Gas,27,2293.0,Fossil
23,France,Europe,Coal,1,120.0,Fossil


In [6]:

# Create the fossil fuel treemap

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    title='Fossil Fuel Energy Consumption by Region and Country'
)

fig.show()


In [7]:

# Define CVD-safe colours
fossil_colors = {
    'Coal': '#2E75B6',
    'Oil': '#E07B39',
    'Natural Gas': '#F5C842'
}

# Create the treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title='Fossil Fuel Energy Consumption by Region and Country',
    height=700,
    template='plotly_white'
)

fig.show()


In [8]:

# Make Region and Country parent nodes grey
for trace in fig.data:
    trace.marker.colors = [
        fossil_colors[label] if label in fossil_colors else '#BDBDBD'
        for label in trace.labels
    ]

# Display energy consumption in TWh
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    textinfo='text',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
    root_color='#FFFFFF'
)

fig.show()


In [9]:

# Calculate total fossil energy consumption by region
region_totals = (
    df_fossil.groupby('Region')['TWh']
    .sum()
    .sort_values(ascending=False)
)

# Display the results
print(region_totals)


Region
Asia             35045.1
Europe           15387.0
Latin America    12376.0
North America    11131.0
Middle East       9671.0
Africa            8620.0
Oceania           8142.0
Name: TWh, dtype: float64


In [10]:

# Identify the region with the highest consumption
top_region = region_totals.idxmax()
top_value = region_totals.max()

print("Highest fossil energy consumption:", top_region)
print(f"Total consumption: {top_value:,.0f} TWh")


Highest fossil energy consumption: Asia
Total consumption: 35,045 TWh


In [11]:

# Add an insight-based title
fig.update_layout(
    title=dict(
        text=(
            f"{top_region} Has the Highest Fossil Fuel "
            f"Consumption ({top_value:,.0f} TWh)"
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),
    margin=dict(t=90, l=20, r=20, b=20)
)

fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [12]:

# Task 2 — Filter low-carbon energy

df_lowcarbon = df.loc[
    df['Source_Type'] == 'Low-carbon'
].copy()

# Calculate total low-carbon TWh by country
lowcarbon_country = (
    df_lowcarbon.groupby('Country', as_index=False)['TWh']
    .sum()
)

# Sort countries from highest to lowest
lowcarbon_country = lowcarbon_country.sort_values(
    'TWh',
    ascending=False
)

# Display the first 10 countries
print(lowcarbon_country.head(10))


           Country     TWh
4           France  7593.0
9           Norway  6174.0
2           Canada  5969.0
1           Brazil  4975.0
11     South Korea  2484.0
6            India  1336.0
13   United States  1137.0
7            Japan  1012.0
12  United Kingdom   993.0
5          Germany   929.0


In [14]:

# Create a root category
lowcarbon_country['Category'] = 'Low-carbon'


In [15]:

# Task 2 — Low-carbon treemap

fig_tree = px.treemap(
    lowcarbon_country,
    path=['Category', 'Country'],
    values='TWh',
    title='Low-Carbon Energy Consumption by Country',
    color='TWh',
    color_continuous_scale='Blues',
    height=650
)

fig_tree.show()


In [18]:

# Identify the leading country
leading_country = lowcarbon_country.iloc[0]['Country']

# Add an insight-based title
fig_bar.update_layout(
    title=f"{leading_country} Leads in Low-Carbon Energy Consumption"
)


In [17]:

# Task 2 — Horizontal bar chart

fig_bar = px.bar(
    lowcarbon_country,
    x='TWh',
    y='Country',
    orientation='h',
    title='Low-Carbon Energy Consumption by Country',
    labels={
        'TWh': 'Energy Consumption (TWh)',
        'Country': 'Country'
    },
    color_discrete_sequence=['#2E75B6'],
    height=700
)

# Show highest-consuming countries at the top
fig_bar.update_yaxes(
    categoryorder='total ascending'
)

fig_bar.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE* The horizontal bar chart is more effective for comparing low-carbon energy consumption rankings across countries because all bars share a common baseline, making differences in TWh easier to compare. Countries are also sorted from highest to lowest, so the ranking is immediately clear.

The treemap is useful for showing each country's contribution to total low-carbon energy consumption, as larger rectangles represent greater consumption. However, comparing rectangle areas is more difficult than comparing bar lengths, especially when countries have similar values.

Therefore, I would choose the horizontal bar chart for comparing country rankings and the treemap for understanding part-to-whole relationships.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [19]:

# Task 3 — Aggregate fossil energy by region and source

region_source = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# Display the aggregated data
print(region_source)


           Region       Source      TWh
0          Africa         Coal   6727.0
1          Africa  Natural Gas    472.0
2          Africa          Oil   1421.0
3            Asia         Coal  17301.1
4            Asia  Natural Gas   8244.0
5            Asia          Oil   9500.0
6          Europe         Coal    887.0
7          Europe  Natural Gas   7785.0
8          Europe          Oil   6715.0
9   Latin America         Coal   1261.0
10  Latin America  Natural Gas   4039.0
11  Latin America          Oil   7076.0
12    Middle East  Natural Gas   4573.0
13    Middle East          Oil   5098.0
14  North America         Coal   1565.0
15  North America  Natural Gas   4031.0
16  North America          Oil   5535.0
17        Oceania         Coal   2919.0
18        Oceania  Natural Gas   2599.0
19        Oceania          Oil   2624.0


In [20]:

# Calculate total fossil energy consumption per region
region_source['Region_Total'] = (
    region_source.groupby('Region')['TWh']
    .transform('sum')
)

# Calculate each fuel's percentage share
region_source['Share_pct'] = (
    region_source['TWh'] /
    region_source['Region_Total']
) * 100

# Display the results
print(region_source.round(2))


           Region       Source      TWh  Region_Total  Share_pct
0          Africa         Coal   6727.0        8620.0      78.04
1          Africa  Natural Gas    472.0        8620.0       5.48
2          Africa          Oil   1421.0        8620.0      16.48
3            Asia         Coal  17301.1       35045.1      49.37
4            Asia  Natural Gas   8244.0       35045.1      23.52
5            Asia          Oil   9500.0       35045.1      27.11
6          Europe         Coal    887.0       15387.0       5.76
7          Europe  Natural Gas   7785.0       15387.0      50.59
8          Europe          Oil   6715.0       15387.0      43.64
9   Latin America         Coal   1261.0       12376.0      10.19
10  Latin America  Natural Gas   4039.0       12376.0      32.64
11  Latin America          Oil   7076.0       12376.0      57.18
12    Middle East  Natural Gas   4573.0        9671.0      47.29
13    Middle East          Oil   5098.0        9671.0      52.71
14  North America        

In [21]:

# Check that percentages add up to 100% per region
percentage_check = (
    region_source.groupby('Region')['Share_pct']
    .sum()
)

print(percentage_check.round(2))


Region
Africa           100.0
Asia             100.0
Europe           100.0
Latin America    100.0
Middle East      100.0
North America    100.0
Oceania          100.0
Name: Share_pct, dtype: float64


In [22]:

# Task 3 — Horizontal 100% stacked bar chart

fig_stack = px.bar(
    region_source,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    color_discrete_map={
        'Coal': '#2E75B6',
        'Oil': '#E07B39',
        'Natural Gas': '#F5C842'
    },
    title='Fossil Fuel Composition by Region',
    labels={
        'Share_pct': 'Share of Fossil Energy (%)',
        'Region': 'Region',
        'Source': 'Energy Source'
    },
    height=550
)

fig_stack.update_xaxes(range=[0, 100])

fig_stack.show()


In [23]:

# Display percentage labels inside each segment
fig_stack.update_traces(
    texttemplate='%{x:.1f}%',
    textposition='inside',
    insidetextanchor='middle'
)

fig_stack.update_layout(
    uniformtext_minsize=10,
    uniformtext_mode='hide'
)


In [24]:

# Select coal records only
coal_data = region_source[
    region_source['Source'] == 'Coal'
]

# Find the region with the highest coal share
coal_data = coal_data.sort_values(
    'Share_pct',
    ascending=False
)

# Display coal shares by region
print(coal_data[['Region', 'Share_pct']].round(2))

# Identify the leading region
top_coal_region = coal_data.iloc[0]['Region']
top_coal_share = coal_data.iloc[0]['Share_pct']

print(
    f"\n{top_coal_region} has the highest coal share: "
    f"{top_coal_share:.1f}%"
)


           Region  Share_pct
0          Africa      78.04
3            Asia      49.37
17        Oceania      35.85
14  North America      14.06
9   Latin America      10.19
6          Europe       5.76

Africa has the highest coal share: 78.0%


In [25]:

# Update chart with an insight-based title
fig_stack.update_layout(
    title=dict(
        text=(
            f"{top_coal_region} Has the Highest Coal Share "
            f"({top_coal_share:.1f}%)"
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),
    template='plotly_white',
    margin=dict(t=90, l=20, r=20, b=20)
)


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The 100% stacked bar chart shows the proportion of coal, oil, and natural gas in each region's total fossil energy consumption.

Based on the results, Africa has the highest coal share, at approximately 78.0%. This means coal makes up the largest proportion of its fossil fuel consumption compared with the other regions.

The 100% stacked bar chart is useful because every region is represented by a bar of equal length, making it easier to compare the proportions of different energy sources. However, it does not show the actual amount of energy consumed in TWh.

Therefore, this visualization is more suitable for comparing fossil fuel composition and relative dependency than for comparing total energy consumption.

In [26]:

# Sort regions by coal share
coal_order = coal_data['Region'].tolist()

fig_stack = px.bar(
    region_source,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    category_orders={
        'Region': coal_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map={
        'Coal': '#2E75B6',
        'Oil': '#BDBDBD',
        'Natural Gas': '#E0E0E0'
    },
    labels={'Share_pct': 'Share of Fossil Energy (%)'},
    title=f"{top_coal_region} Has the Highest Coal Share ({top_coal_share:.1f}%)",
    height=550
)

# Show labels on coal only
fig_stack.update_traces(
    texttemplate='%{x:.1f}%',
    textposition='inside',
    selector=dict(name='Coal')
)

fig_stack.update_yaxes(
    categoryorder='array',
    categoryarray=coal_order[::-1]
)

fig_stack.update_xaxes(range=[0, 100])

fig_stack.show()
